# IFEval — instruction following (local)

IFEval gives the model 541 prompts that contain verifiable instructions ("write at least 300 words", "use no commas", "end with the phrase ...") and checks each response with code.

This notebook runs the benchmark **locally**, straight against the model's chat
endpoint — no EvalHub and no OpenShift. It is self-contained: all settings are in the
configuration cell below, and no job file is needed. It is the notebook equivalent of
the matching `run-*-local.sh` script.

**How it works:** [lm-evaluation-harness](https://github.com/EleutherAI/lm-evaluation-harness)
(`lm_eval`) is run through `uvx`, which fetches it on demand into a throwaway
environment. Questions are sent to the model's `/chat/completions` endpoint with the
model's chat template applied, and answers are scored locally.

**Requires:** [`uv`](https://docs.astral.sh/uv/) (for `uvx`) and network access to the
model endpoint.

## 1. Configuration

Everything you might want to change lives in this one cell.

In [1]:
import json, os, sys, shutil, subprocess
from pathlib import Path

# eval_env.py sits next to the notebooks (the kernel may have started in the parent folder)
sys.path.insert(0, str(next(d for d in (Path("."), Path("notebooks")) if (d / "eval_env.py").is_file())))
from eval_env import setup

# Model endpoint and optional proxy, from .env (or prompts); see eval_env.py
MODEL_URL, MODEL_NAME, MODEL_API_KEY = setup()


# --- Benchmark ---
TASK = "ifeval"  # lm-eval task name
LM_EVAL_VERSION = "0.4.12"  # pinned so results are repeatable
LM_EVAL_EXTRAS = "api,ifeval"  # pip extras lm-eval needs for this task
PYPI_INDEX = "https://pypi.org/simple"  # package index used to install lm-eval, instead of any index set in your uv config
NUM_CONCURRENT = 8          # parallel requests to the model
THRESHOLD = 0.6  # minimum score to count as PASS
LIMIT = None                # e.g. 10 for a quick smoke test; None runs the full benchmark


Loaded settings from .env


## Helper

`run()` launches a command and streams its output into the notebook as it arrives, so you
can watch progress. It raises an error if the command exits non-zero.

In [2]:
def run(cmd, env=None):
    proc = subprocess.Popen(cmd, env={**os.environ, **(env or {})}, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"{cmd[0]} exited with {proc.returncode}")

## 2. Run the benchmark

Builds the `lm_eval` command and runs it. Completed requests are cached on disk, so if
the run dies partway (for example the gateway drops a connection), re-running this cell
resumes where it left off. The cache is deleted after a successful run so the next run
always gets fresh answers from the model.

In [3]:
OUTPUT = Path(f".eval-notebook-{TASK}")             # lm-eval writes results and samples here
CACHE = Path(f".eval-notebook-{TASK}-cache") / "requests"
CACHE.parent.mkdir(exist_ok=True)
shutil.rmtree(OUTPUT, ignore_errors=True)

cmd = ["uvx", "--default-index", PYPI_INDEX, "--python", "3.12", "--from", f"lm-eval[{LM_EVAL_EXTRAS}]=={LM_EVAL_VERSION}",
       "lm_eval", "--model", "local-chat-completions",
       "--model_args", f"model={MODEL_NAME},base_url={MODEL_URL.rstrip('/')}/chat/completions,"
                       f"num_concurrent={NUM_CONCURRENT},max_retries=5,tokenized_requests=False",
       "--tasks", TASK, "--apply_chat_template", "--log_samples",
       "--output_path", str(OUTPUT), "--use_cache", str(CACHE)]
if LIMIT:
    cmd += ["--limit", str(LIMIT)]

# Clear uv index overrides inherited from the environment so lm-eval installs from PYPI_INDEX.
uv_env = {k: "" for k in ("UV_INDEX", "UV_INDEX_URL", "UV_EXTRA_INDEX_URL", "UV_DEFAULT_INDEX")}
run(cmd, env={**uv_env, "OPENAI_API_KEY": MODEL_API_KEY or "not-required"})
shutil.rmtree(CACHE.parent, ignore_errors=True)

Installed 73 packages in 23.46s
Bytecode compiled 6548 files in 21.47s
2026-11-20:15:28:15 INFO     [config.evaluate_config:307] Using default fewshot_as_multiturn=True.
2026-11-20:15:28:59 INFO     [_cli.run:388] Selected Tasks: ['ifeval']
2026-11-20:15:28:59 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-11-20:15:28:59 INFO     [evaluator:239] Initializing local-chat-completions model, with arguments: {'model': 'gemma-4', 'base_url': 'https://maas.apps.ocp.home.glroland.com/sandbox/gemma-4/v1/chat/completions', 'num_concurrent': 8, 'max_retries': 5, 'tokenized_requests': False}
2026-11-20:15:29:00 INFO     [models.api_models:179] Using max length 2048 - 1
2026-11-20:15:29:00 INFO     [models.api_models:200] Using tokenizer None
2026-11-20:15:29:00 INFO     [evaluator:278] Using cache at .eval-notebook-ifeval-cache/requests_rank0.db
/tmp/.tmp8BcLh1/archive-v0/XBpRiTY4JPY82AWp

## 3. Score and pass/fail

The headline score is `inst_level_strict_acc` — the fraction of individual instructions followed exactly. The loose and prompt-level variants are printed too. Per-question model answers are in the `.eval-notebook-<task>` folder (`samples_*.jsonl`) if you want to see what it got wrong.

In [4]:
results_file = next(OUTPUT.rglob("results_*.json"))
scores = json.loads(results_file.read_text())["results"][TASK]
print(json.dumps({k.split(",")[0]: v for k, v in scores.items()
                  if k.endswith(",none") and not k.startswith("alias")}, indent=2))

score = scores["inst_level_strict_acc,none"]
print("PASS" if score >= THRESHOLD else "FAIL", f"- inst_level_strict_acc {score:.3f} vs threshold {THRESHOLD}")

{
  "prompt_level_strict_acc": 0.8835489833641405,
  "prompt_level_strict_acc_stderr": 0.01380352728407129,
  "inst_level_strict_acc": 0.9172661870503597,
  "inst_level_strict_acc_stderr": "N/A",
  "prompt_level_loose_acc": 0.8946395563770795,
  "prompt_level_loose_acc_stderr": 0.013211920861725028,
  "inst_level_loose_acc": 0.9280575539568345,
  "inst_level_loose_acc_stderr": "N/A"
}
PASS - inst_level_strict_acc 0.917 vs threshold 0.6


## 4. Results dashboard

The headline score hides a lot. These charts show the same run four ways: strict vs. loose scoring, how difficulty grows with the number of instructions in a prompt, which of the instruction types the model is weakest at (with 95% confidence intervals), and the actual prompts it failed.

In [ ]:
import sys
from pathlib import Path
# eval_report.py sits next to the notebooks (the kernel may have started in the parent folder)
sys.path.insert(0, str(next(d for d in (Path("."), Path("notebooks")) if (d / "eval_report.py").is_file())))
from eval_report import report_ifeval

report_ifeval(OUTPUT, THRESHOLD)